# ICD-O Diagnosis Group Mapper: Development & Testing Notebook

This notebook serves as the development environment for a Python-based script designed to map ICD-O 4-digit diagnosis codes (e.g., 9380/3) or diagnosis terms (e.g., Glioma, malignant) to their corresponding diagnosis groups.

---
<center>
  <h3>Input Example (2 columns) → Output Example (4 columns)</h1>
</center>

<table align="center">
<tr>
<td>

| code   | term              |
|--------|-------------------|
| 8000/0 | Neoplasm, benign  |
| 8000/0 | Tumor, benign     |
| 8000/1 | Tumor, NOS        |
| 8010/0 | Epithelial tumor  | 
</td>
<td align="center" style="font-size: 24px; padding: 20px;">

**→**

</td>
<td>

| code   | term              | prefix | group           |
|--------|-------------------|--------|-----------------|
| 8000/0 | Neoplasm, benign  | 800    | Neoplasms, NOS  |
| 8000/0 | Tumor, benign     | 800    | Neoplasms, NOS  |
| 8000/1 | Tumor, NOS        | 800    | Neoplasms, NOS  |
| 8010/0 | Epithelial tumor  | 801-804 | Epithelial neoplasms, NOS |
</td>
</tr>
</table>

---
- Auto-Detection: The script automatically identifies whether the user provided codes or terms by checking for the ####/# pattern or matching against a known list of ICD-O terms.

- Intelligent Mapping:
For codes: extracts the first three digits of a diagnosis code to find the parent group.
For terms: performs exact string matching against standardized ICD-O preferred and synonyms terms.

- Data Validation: Validates the input format and reports "Recognition Errors" for any rows that do not match the expected format, allowing for partial mapping. If no match is found, "No diagnosis group match found" is returned in a fifth column titled "Comments". 

- Standardized Output: Produces a four-column TSV containing the original diagnosis code and term alongside the mapped group code/range and group name.

### Possible Additional Features
- complex command-line argument setup that accepts user_input parameter -optional
- rewrite the key if the user wants to specify it (using parameter) -optional

### Data Loading

In [38]:
# load packages 
import pandas as pd
import re
from datetime import datetime
import glob
import os
import sys
from pathlib import Path
# set up error-handling
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

In [39]:
# load files
code_mapping_file = pd.read_csv("../data/code_mapping_file.csv") # doesn't have synonyms (if the user supplies codes)
term_mapping_file = pd.read_csv("../data/term_mapping_file.csv") # has synonyms (if the user supplies terms)

# look for any csv, tsv, or excel file in the input folder
input_files = glob.glob("../input/*.[ct]sv") + glob.glob("../input/*.xlsx")

if not input_files:
    logger.error("No input files found in the /input directory.")
    sys.exit()
else:
    file_to_process = input_files[0]
    logger.info(f"Processing: {os.path.basename(file_to_process)}")
    user_input = pd.read_csv(file_to_process, sep=None, engine='python')

2026-03-03 10:24:10,272 - INFO - Processing: example.csv


In [40]:
user_input.head()

,code,term
0,8000/0,"Neoplasm, benign"
1,8000/0,"Tumor, benign"
2,8000/0,"Unclassified tumor, benign"
3,8000/1,"Neoplasm, uncertain whether benign or malignant"
4,8000/1,"Neoplasm, NOS"


### Data Validation

In [41]:
data = user_input.iloc[:,0].astype(str).str.strip()
invalid_rows = []

code_pattern = r'^\d{4}/\d$'
term_list = term_mapping_file['term']

# if the user supplies codes
if data.str.match(code_pattern).any():
    mapping_file = code_mapping_file
    key = 'code'
    user_input = user_input.rename(columns={user_input.columns[0]: key})
    if data.str.match(code_pattern).all():
        logger.info("✅ Validation Success: All entries are codes.")
    else:
        invalid_rows = data[~data.str.match(code_pattern)].tolist()
        logger.warning(f"❌ Recognition Error: Some rows do not follow ####/# format.")
        
# if the user supplies terms
elif data.isin(term_list).any():
    mapping_file = term_mapping_file
    key = 'term'
    user_input = user_input.rename(columns={user_input.columns[0]: key})
    if data.isin(term_list).all():
        logger.info("✅ Validation Success: All entries are terms.")
    else: 
        invalid_rows = data[~data.isin(term_list)].tolist()
        logger.warning(f"❌ Recognition Error: Some terms do not match ICD-O exactly.")
        
# if user-supplied data is not recognized
else:
    logger.error(f"❌ Recognition Error: No entries were recognized.")
    logger.error(f" Codes must follow the format ####/#. Terms must match ICD-O terms exactly (including commas/caps/spaces).")       

if invalid_rows:
    logger.warning(f"Some entries were not recognized: {invalid_rows}") 

2026-03-03 10:24:10,411 - INFO - ✅ Validation Success: All entries are codes.


In [42]:
# if second column is supplied
if len(user_input.columns) > 1:
    col2_data = user_input.iloc[:, 1].astype(str).str.strip()
    col2_name = user_input.columns[1]

    if key == 'code' and col2_data.isin(term_list).any():
        user_input = user_input.rename(columns={col2_name: 'term'})
        logger.info("✅ Column 2 recognized as terms and renamed.")
    
    elif key == 'term' and col2_data.str.match(code_pattern).any():
        user_input = user_input.rename(columns={col2_name: 'code'})
        logger.info("✅ Column 2 recognized as codes and renamed.")
    
    else:
        logger.warning(f"Column 2 ('{col2_name}') not recognized as ICD-O data. Keeping as-is.")

2026-03-03 10:24:10,454 - INFO - ✅ Column 2 recognized as terms and renamed.


### Group Information Lookup

In [43]:
# look up the group information
output = user_input.merge(
    mapping_file[['code', 'term', 'range', 'group']],
    on = key,
    how = 'left'
    )

In [44]:
# remove duplicate columns, keeping user's terms
cols_to_drop = [col for col in output.columns if col.endswith('_y')]
output = output.drop(columns=cols_to_drop)
output.columns = [col.replace('_x', '') for col in output.columns]

In [49]:
output.head(3)

,code,term,range,group
0,8000/0,"Neoplasm, benign",800,"Neoplasms, NOS"
1,8000/0,"Tumor, benign",800,"Neoplasms, NOS"
2,8000/0,"Unclassified tumor, benign",800,"Neoplasms, NOS"


### Reporting

In [46]:
total = len(output)
# add comments column for unmatched entries
unmatched = len(output[output['group'].isna()])
if unmatched > 0:
    output['Comments'] = ""
    output.loc[output['group'].isna(), 'Comments'] = "No diagnosis group match found"

In [47]:
logger.info(f"\nSummary:")
logger.info(f"Total entries: {total}")
logger.info(f"Successfully mapped: {total - unmatched}")
logger.info(f"Unmatched: {unmatched}")

2026-03-03 10:24:10,619 - INFO - 
Summary:
2026-03-03 10:24:10,629 - INFO - Total entries: 50
2026-03-03 10:24:10,631 - INFO - Successfully mapped: 50
2026-03-03 10:24:10,635 - INFO - Unmatched: 0


### Saving

In [48]:
# generate a timestamp string 
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# create output file name
tsv_filename = f'../output/output_{timestamp}.tsv'

# export the file
output.to_csv(tsv_filename, sep='\t', index=False)
logger.info(f"✅ Files exported successfully: {tsv_filename}")

2026-03-03 10:24:10,673 - INFO - ✅ Files exported successfully: ../output/output_20260303_1024.tsv


In [ ]:
# for future script
# lets user pass a filename in terminal
#if __name__ == "__main__":
    #input_file = sys.argv[1] 